# Données de la chaîne — inventaire exécutable

Carte d'identité de chaque fichier de données important du banc de
génération AP-HP : **documentation** (rôle, producteur, emplacement, ce qui
fait foi, consommation), **vérification à la demande** (les contrôles de la
chaîne sont rejoués, pas réécrits) et **exploration libre** (une cellule
vide par fiche).

**Règle : ce qui fait foi reste les contrats et `SCHEMA_SOURCE` — ce
notebook les exécute.** Il n'y a ici aucun schéma redéclaré, aucun contrôle
dupliqué : chaque fiche appelle l'existant (`bench.banc.verifier_source`,
`bench.fiches.charger_index`, le loader fictomed, le script de
substitution…). Si un contrôle change, c'est dans `bench/` qu'il change ;
la fiche le reflète à la prochaine exécution.

Chaque fiche est **autonome** (exécutable seule après la cellule d'imports)
et **tolérante** : fichier absent → message qui dit comment l'obtenir,
jamais d'exception. Aucune clé API n'est nécessaire.

Fiches : 1. parquet de scénarios courant · 2. référence de substitution des
DP imprécis · 3. dictionnaire des spécialités par racine · 4. mapping
`type_unite` · 5. librairie de fiches recode-icd · 6. référentiels fictomed
annexes · 7. journal `usage_log.csv` · 8. anatomie d'un dossier de run.

In [ ]:
# --- Imports et chemins — à exécuter en premier ; chaque fiche est ensuite autonome ---
import sys
from pathlib import Path

REPO_ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                  if (p / "bench").is_dir() and (p / "core").is_dir()), None)
assert REPO_ROOT, "Racine du repo Stream introuvable — lancer le notebook depuis generation/."
for _p in (REPO_ROOT, REPO_ROOT / "scripts"):
    if str(_p) not in sys.path:
        sys.path.append(str(_p))

import polars as pl
from IPython.display import display

from bench import BenchError, charger_index, codes_emissibles, deriver_agean, scenario_dirs
from bench.banc import DATA_APHP, REFERENTIALS, TESTS_DIR, USAGE_LOG, verifier_source
from enrichissement import Politique

pl.Config.set_tbl_rows(40)
pl.Config.set_fmt_str_lengths(70)


def present(chemin: Path, comment_obtenir: str) -> bool:
    """Tolérance : dit si le fichier (ou dossier) est là, sinon comment
    l'obtenir — jamais d'exception."""
    chemin = Path(chemin)
    if chemin.exists():
        if chemin.is_file():
            taille = chemin.stat().st_size
        else:
            taille = sum(f.stat().st_size for f in chemin.rglob("*") if f.is_file())
        rel = chemin.relative_to(REPO_ROOT) if chemin.is_relative_to(REPO_ROOT) else chemin
        lisible = (f"{taille / 1e6:.1f} Mo" if taille >= 1e6
                   else f"{taille / 1e3:.0f} Ko" if taille >= 1e3 else f"{taille} o")
        print(f"PRÉSENT  {rel} — {lisible}")
        return True
    print(f"ABSENT   {chemin}\n         → pour l'obtenir : {comment_obtenir}")
    return False


print("Racine :", REPO_ROOT)
print("Données :", DATA_APHP, "| référentiels :", REFERENTIALS, "| runs :", TESTS_DIR)

## 1. Le parquet de scénarios courant

- **Rôle** : la source des scénarios — un séjour PMSI agrégé par ligne
  (DP, DAS, sexe, classe d'âge, GHM, durée, modes…), d'où le banc tire le
  pool candidat que fictomed transforme en scénarios cliniques.
- **Producteur, évolution** : le projet amont `scenarios_bn_pmsi`, par
  campagne (`scenarios_C1.parquet`, puis C2…). L'âge exact n'est pas livré
  (agrégé en classes `cage`, protection assumée) : `agean` est **dérivée**
  par la chaîne. La version `_dp` est l'artefact de
  `scripts/substituer_dp_imprecis.py` (DP imprécis remplacés, colonnes de
  traçabilité `dp_origine` / `dp_substitue` / `repli_substitution`).
- **Emplacement** : `data/aphp/` (non versionné) ; le notebook de
  génération le désigne par `SOURCE_PROFILES_PATH`.
- **Ce qui fait foi** : `bench.banc.SCHEMA_SOURCE` (exigences par colonne,
  documentées sur ce que la chaîne consomme réellement) et son contrôle
  `verifier_source` ; la dérivation d'âge est `bench.scenarios.deriver_agean`.
- **Consommation** : `preparer_pool` — contrôle, dérivation d'`agean`,
  réparation de `racine`, typologie, filtre DP, tirage, spécialité
  (service), enrichissement, contrôle des fiches.

In [ ]:
SOURCES = [DATA_APHP / "scenarios_C1.parquet", DATA_APHP / "scenarios_C1_dp.parquet"]
OBTENIR = ("corpus de campagne déposé par Rémi dans data/aphp/ (projet amont scenarios_bn_pmsi) ; "
           "la version _dp sort de `python scripts/substituer_dp_imprecis.py <corpus> "
           "--ref data/aphp/ref_substitution_imprecis.parquet`")
politique = Politique()  # les décisions d'enrichissement (âge minimal, préfixes exclus)

for chemin in SOURCES:
    print("=" * 100)
    if not present(chemin, OBTENIR):
        continue
    df = pl.read_parquet(chemin)
    # Le contrôle de la chaîne, tel quel (SCHEMA_SOURCE) — imprime son rapport
    try:
        verifier_source(chemin, df=df)
    except BenchError:
        print("→ NON CONFORME : la chaîne refusera ce fichier (écarts ci-dessus).")
    print(f"\n{df.height} lignes × {df.width} colonnes")
    display(df.head(3))
    if "branche" in df.columns:
        display(df.group_by("branche").len().sort("branche"))
    if {"TPEC", "DPEC"} <= set(df.columns):
        display(df.group_by("TPEC", "DPEC").len().sort(["TPEC", "DPEC"]))
    else:
        print("Typologie TPEC/DPEC non fournie : preparer_pool la calcule (bench.scenarios.with_typologie).")
    if "dp_substitue" in df.columns:
        par = "branche" if "branche" in df.columns else None
        stats = (df.group_by(par) if par else df.group_by(pl.lit("(toutes)").alias("branche"))).agg(
            pl.len().alias("lignes"),
            (pl.col("dp_substitue").mean() * 100).round(1).alias("dp_substitues_%"),
            (pl.col("repli_substitution") == 0).sum().alias("repli_0"),
            (pl.col("repli_substitution") == 1).sum().alias("repli_1"),
            (pl.col("repli_substitution") == 2).sum().alias("repli_2"),
        ).sort("branche")
        display(stats)
        rapport = chemin.with_suffix(".rapport.txt")
        print("rapport de substitution :", rapport if rapport.is_file() else "(absent — écrit par le script à côté de la sortie)")
    # Part enrichissable — mêmes ingrédients que l'enrichisseur : l'âge que la
    # chaîne verra (deriver_agean si agean manque) et la Politique (age_min,
    # prefixes_exclusion sur DP + DAS). L'enrichisseur reste seul juge ligne à ligne.
    if {"diag2", "diagnostic_associes"} <= set(df.columns) and ("agean" in df.columns or {"cage", "id_scenario"} <= set(df.columns)):
        age = df["agean"] if "agean" in df.columns else deriver_agean(df)[0]["agean"]
        motif_exclusion = "(^| )(" + "|".join(politique.prefixes_exclusion) + ")"
        exclu = (pl.col("diag2").fill_null("").str.contains(motif_exclusion)
                 | pl.col("diagnostic_associes").fill_null("").str.contains(motif_exclusion))
        part = df.with_columns(age.alias("_age")).select(
            ((pl.col("_age") >= politique.age_min) & ~exclu).mean()).item()
        print(f"Part enrichissable (âge ≥ {politique.age_min}, hors préfixes {politique.prefixes_exclusion}) : "
              f"{part:.1%}" + ("" if "agean" in df.columns else " — sur l'âge DÉRIVÉ de cage"))

In [ ]:
# Exploration — parquet de scénarios

## 2. Référence de substitution des DP imprécis

- **Rôle** : l'agrégat seuillé qui dit, par catégorie CIM-10 et strate
  (classe d'âge, sexe), quels codes sont observés dans le PMSI réel avec
  quels effectifs, et lesquels sont « sans précision ». C'est à la fois la
  liste de détection des DP imprécis et la loi de tirage des remplaçants.
- **Producteur, évolution** : export de la plateforme (projet amont) ;
  un fichier par kit / campagne. `niveau` (sévérité CMA) est livré mais
  non utilisé en v1.
- **Emplacement** : `data/aphp/ref_substitution_imprecis.parquet` (non
  versionné ; le chemin est libre, passé en `--ref`).
- **Ce qui fait foi** : l'en-tête de `scripts/substituer_dp_imprecis.py`
  (colonnes attendues `COLONNES_REF`, règle, repli, déterminisme) ; sa
  fonction `charger_reference` vérifie et normalise — elle est appelée ici.
- **Consommation** : `scripts/substituer_dp_imprecis.py`, étape
  fichier → fichier en amont du banc.

In [ ]:
chemin = DATA_APHP / "ref_substitution_imprecis.parquet"
if present(chemin, "export de la plateforme (projet amont scenarios_bn_pmsi), à déposer dans data/aphp/"):
    from substituer_dp_imprecis import COLONNES_REF, charger_reference  # scripts/ — mêmes contrôles que le script

    ref = charger_reference(chemin)
    print("colonnes attendues :", COLONNES_REF)
    print(ref.schema)
    display(ref.head(5))
    imprecis = ref.filter(pl.col("imprecis"))
    n_codes = ref["code"].n_unique()
    print(f"{ref.height} lignes — {ref['cat'].n_unique()} catégories couvertes — "
          f"{n_codes} codes distincts dont {imprecis['code'].n_unique()} imprécis "
          f"({imprecis['code'].n_unique() / n_codes:.1%}) ; lignes imprécises : {ref['imprecis'].mean():.1%}")
    sans_precis = ref.group_by("cat").agg((~pl.col("imprecis")).sum().alias("n_precis")).filter(pl.col("n_precis") == 0)
    print(f"catégories sans aucun code précis (les DP y seront conservés) : {sans_precis.height}")
    print("distribution des effectifs nb :")
    display(ref["nb"].describe())
    print("niveau (sévérité CMA, non utilisé en v1) :", ref["niveau"].value_counts().sort("niveau").to_dicts())
    rapport = DATA_APHP / "scenarios_C1_dp.rapport.txt"
    if rapport.is_file():
        print("\n--- dernier rapport de substitution (", rapport.name, ") ---")
        print(rapport.read_text(encoding="utf-8"))

In [ ]:
# Exploration — référence de substitution

## 3. Dictionnaire des spécialités par racine de GHM

- **Rôle** : pour une racine de GHM et une classe d'âge (`ge_18` /
  `lt_18`), la répartition des spécialités d'unité médicale (`lib_spe_uma`)
  observée à l'AP-HP, avec le ratio et les effectifs — fictomed en tire la
  spécialité du service qui « signe » le CRH.
- **Producteur, évolution** : référentiel AP-HP livré avec le dossier
  `data/aphp/referentials/` (mai 2026) ; mis à jour avec les référentiels,
  pas avec les campagnes.
- **Emplacement** : `data/aphp/referentials/dictionnaire_spe_racine.parquet`.
- **Ce qui fait foi** : le loader fictomed (`fictomed.sites.aphp.loader`,
  `load_referentials()["specialty"]`) qui renomme `racine` →
  `drg_parent_code`, `lib_spe_uma` → `specialty`, `ratio_spe_racine` →
  `ratio`. Clé : (`racine`, `age`, `lib_spe_uma`).
- **Consommation** : `bench.scenarios.deriver_specialite` (appelée par
  `preparer_pool` après le tirage) — étage 2 de l'attribution du service :
  jointure sur (`racine` réparée, groupe d'âge dérivé d'`agean`), une
  candidate → elle, plusieurs → tirage pondéré par ratio à graine
  composite par ligne, aucune → repli (pas de ligne Service, le modèle
  propose). Sans colonne `specialty` dans le profil, fictomed applique sa
  propre jointure « première spécialité de la racine », sans groupe d'âge
  ni ratio — le comportement d'avant, source des erreurs constatées.

In [ ]:
chemin = REFERENTIALS / "dictionnaire_spe_racine.parquet"
if present(chemin, "référentiel AP-HP du dossier data/aphp/referentials/ (livraison des référentiels, pas des campagnes)"):
    from fictomed.sites.aphp.loader import load_referentials  # le lecteur fictomed, tel quel

    spe = load_referentials(REFERENTIALS)["specialty"].collect()
    print("schéma (après renommage du loader) :", spe.schema)
    display(spe.head(5))
    cle = ["drg_parent_code", "age", "specialty"]
    print(f"{spe.height} lignes — {spe['drg_parent_code'].n_unique()} racines — "
          f"{spe['specialty'].n_unique()} spécialités — clé {cle} unique : "
          f"{spe.select(cle).n_unique() == spe.height}")
    sommes = spe.group_by("drg_parent_code", "age").agg(pl.col("ratio").sum().alias("somme_ratio"))
    print(f"somme des ratios par (racine, age) : min {sommes['somme_ratio'].min():.3f}, max {sommes['somme_ratio'].max():.3f}")
    display(spe.group_by("age").len().sort("age"))
    # Couverture contre le corpus courant : racines présentes dans le dictionnaire
    corpus = DATA_APHP / "scenarios_C1.parquet"
    if corpus.is_file():
        c1 = pl.read_parquet(corpus, columns=["racine", "ghm2"] + (["branche"] if "branche" in pl.read_parquet_schema(corpus) else []))
        racines = set(spe["drg_parent_code"].to_list())
        c1 = c1.with_columns(
            pl.col("racine").is_in(list(racines)).alias("couverte"),
            pl.col("ghm2").str.slice(0, 5).is_in(list(racines)).alias("couverte_par_ghm2"),
        )
        par = ["branche"] if "branche" in c1.columns else []
        display(c1.group_by(par).agg(
            pl.len().alias("lignes"),
            pl.col("racine").is_null().sum().alias("racine_nulle"),
            pl.col("couverte").sum().alias("racine_couverte"),
            pl.col("couverte_par_ghm2").sum().alias("ghm2[:5]_couvert"),
        ).sort(par) if par else c1.select(pl.len().alias("lignes"), pl.col("racine").is_null().sum().alias("racine_nulle"),
                                          pl.col("couverte").sum().alias("racine_couverte"), pl.col("couverte_par_ghm2").sum().alias("ghm2[:5]_couvert")))
        print("Lecture : une racine nulle (branche courte de C1) est réparée depuis ghm2[:5] par",
              "bench.scenarios.reparer_racine avant la jointure (compteur au récap de preparer_pool,",
              "attendu 0 après la correction amont).")
        # Répartition attendue des sources de spécialité sur tout le corpus — la
        # chaîne elle-même (deriver_agean → reparer_racine → deriver_specialite)
        from bench import charger_mapping_type_unite, deriver_specialite, reparer_racine

        corpus_df = pl.read_parquet(corpus)
        corpus_df = deriver_agean(corpus_df)[0] if "agean" not in corpus_df.columns else corpus_df
        corpus_df, rapport_racine = reparer_racine(corpus_df)
        mapping_path = REFERENTIALS / "mapping_type_unite.yaml"
        mapping = (charger_mapping_type_unite(mapping_path, spe["specialty"].unique().to_list())
                   if mapping_path.is_file() else {})
        corpus_df, rapport_spe = deriver_specialite(corpus_df, pl.read_parquet(chemin), mapping)
        print(rapport_racine.texte())
        print(rapport_spe.texte())
        par = ["branche", "specialite_source"] if "branche" in corpus_df.columns else ["specialite_source"]
        display(corpus_df.group_by(par).len().sort(par))
        display(corpus_df["specialty"].value_counts().sort("count", descending=True).head(12))
    else:
        print("(corpus scenarios_C1.parquet absent : couverture et répartition non calculées)")

In [ ]:
# Exploration — dictionnaire des spécialités

## 4. Mapping `type_unite`

- **Rôle** : la table de correspondance entre les valeurs de `type_unite`
  des corpus de campagne (HC, GERIATRIE, NEONAT, SC, SC-NEONAT, HP, UHCD…)
  et leur interprétation dans la chaîne (exemption UHCD de la substitution
  des DP, contexte de séjour du CRH), avec un **statut par entrée** :
  `proposition` ou `valide`.
- **Producteur, évolution** : décision Rémi, entrée par entrée ; les
  valeurs nouvelles d'une campagne y entrent en `proposition`. En campagne
  2, `type_unite` sera renseigné sur la branche courte (UHCD).
- **Emplacement** : `data/aphp/referentials/mapping_type_unite.yaml`
  (versionné malgré `data/` ignoré : c'est une décision, pas une donnée),
  construit le 24/09/2026 pré-rempli de propositions.
- **Ce qui fait foi** : le fichier lui-même (statuts) ; son lecteur
  `bench.scenarios.charger_mapping_type_unite` ne rend que les entrées
  `valide` hors `DERIVER`, et refuse une spécialité valide hors des 55
  libellés du dictionnaire. Côté substitution des DP, seule la valeur
  `UHCD` est interprétée (`scripts/substituer_dp_imprecis.py`).
- **Consommation** : étage 1 de `deriver_specialite` (via `preparer_pool`) —
  une entrée `proposition` est ignorée, l'étage 2 prend le relais ;
  corriger les statuts suffit à activer.

In [ ]:
import yaml

candidats = [REFERENTIALS / "mapping_type_unite.yaml", REPO_ROOT / "docs" / "mapping_type_unite.yaml"]
chemin = next((c for c in candidats if c.is_file()), candidats[0])
valeurs_c1: dict = {}
corpus = DATA_APHP / "scenarios_C1.parquet"
if corpus.is_file():
    tu = pl.read_parquet(corpus, columns=["branche", "type_unite"])
    valeurs_c1 = {r["type_unite"]: r["len"] for r in tu.group_by("type_unite").len().sort("len", descending=True).to_dicts()}
    display(tu.group_by("branche", "type_unite").len().sort(["branche", "len"], descending=[False, True]))
if present(chemin, "versionné dans le dépôt (git) : `git checkout -- data/aphp/referentials/mapping_type_unite.yaml` ; "
                   "sinon à reconstruire : `entrees: {valeur: {specialite: <libellé du dictionnaire ou DERIVER>, statut: proposition|valide}}`"):
    from bench import charger_mapping_type_unite

    contenu = yaml.safe_load(chemin.read_text(encoding="utf-8")) or {}
    print({k: v for k, v in contenu.items() if k != "entrees"})
    entrees = contenu.get("entrees", {}) or {}
    display(pl.DataFrame([{"type_unite": k, **v} for k, v in entrees.items()]))
    non_couvertes = [v for v in valeurs_c1 if v is not None and v not in entrees]
    print("valeurs de C1 non couvertes :", non_couvertes or "aucune")
    dico = REFERENTIALS / "dictionnaire_spe_racine.parquet"
    vocab = pl.read_parquet(dico)["lib_spe_uma"].unique().to_list() if dico.is_file() else None
    print("entrées APPLIQUÉES par la chaîne (statut valide, hors DERIVER) :",
          charger_mapping_type_unite(chemin, vocab) or "aucune — tout est en proposition")

In [ ]:
# Exploration — mapping type_unite

## 5. La librairie de fiches recode-icd

- **Rôle** : une fiche descriptive par code CIM-10 (périmètre,
  localisations, exclusions, formulations…), insérée dans le user prompt de
  chaque scénario pour que le modèle sache ce que couvre chaque code ;
  une bibliothèque de fiches exactes et une de fiches de catégorie (repli).
- **Producteur, évolution** : `24p11/recode-icd`, livré en archive
  `recode-icd_fiches_generation_<commit>_<kit>.tar.gz` sous **contrat
  d'interface** ; mise à jour = décision explicite, redéploiement complet,
  trace dans `DEPLOIEMENT.txt`.
- **Emplacement** : `data/aphp/referentials/cards_library/` et
  `cards_library_categories/` (non versionnés), `DEPLOIEMENT.txt` à côté.
- **Ce qui fait foi** : le contrat
  (`https://github.com/24p11/recode-icd/blob/main/docs/livraison/CONTRAT.md`,
  exemplaire épinglé `CONTRAT.md` dans chaque bibliothèque) ; côté
  consommateur, `bench/fiches.py` est le point d'accès unique (l'index
  `index.csv` fait foi, `format_version` connue, classe `emissible`).
- **Consommation** : fictomed lit les fiches (registre `CodeCardsRegistry`)
  au seeding ; `preparer_pool` journalise les codes du pool sans fiche et
  vérifie que les codes ajoutés sont émissibles ; `verifier_environnement`
  sonde le lecteur.

In [ ]:
trace = REFERENTIALS / "DEPLOIEMENT.txt"
if present(trace, "écrit à la main lors du déploiement d'une archive de livraison (archive, commit, kit, date)"):
    print(trace.read_text(encoding="utf-8"))

for nom in ("cards_library", "cards_library_categories"):
    print("=" * 100)
    lib = REFERENTIALS / nom
    if not present(lib, "déployer l'archive de livraison recode-icd (voir DEPLOIEMENT.txt et le contrat)"):
        continue
    print("CONTRAT.md épinglé :", (lib / "CONTRAT.md").is_file())
    try:
        index = charger_index(lib)  # bench.fiches : l'index fait foi, noyau et format_version vérifiés
    except BenchError as exc:
        print("→ REFUSÉE par bench.fiches :", exc)
        continue
    print(f"{index.height} fiches à l'index — format_version {index['format_version'].unique().to_list()} — colonnes : {index.columns}")
    if "classe_generation" in index.columns:
        display(index.group_by("classe_generation").len().sort("classe_generation"))
        print("codes émissibles (tirage / génération) :", len(codes_emissibles(index)))
    if "chapter" in index.columns:
        display(index.group_by("chapter").len().sort("chapter"))
    display(index.head(3))

# Sonde du LECTEUR fictomed (le registre lit-il la librairie déployée ?) — elle
# vit dans bench.banc.verifier_environnement, avec l'assertion fictomed ;
# décommenter pour la rejouer ici :
# from bench.banc import verifier_environnement
# verifier_environnement()

In [ ]:
# Exploration — librairie de fiches

## 6. Les référentiels fictomed annexes

- **Rôle** : les tables que fictomed charge depuis
  `data/aphp/referentials/` pour habiller un scénario — établissements
  (`chu`), spécialités par racine (fiche 3), CIM et CCAM officielles,
  synonymes, listes de codes par situation (surveillance, antécédents,
  séances…), statistiques de GHM, référentiels cancer, prénoms.
- **Producteur, évolution** : dossier de référentiels AP-HP (mai 2026),
  versionné à part des campagnes ; fictomed en fixe la liste dans
  `loader.load_referentials`.
- **Emplacement** : `data/aphp/referentials/` (non versionné).
- **Ce qui fait foi** : le loader fictomed (`fictomed.sites.aphp.loader`),
  clé par clé — la fiche l'appelle et dit ce qui est présent.
- **Consommation** : fictomed au seeding (`servers.yaml` écrit par
  `bench.scenarios.write_fictomed_config` pointe ce dossier).

In [ ]:
from fictomed.sites.aphp.loader import load_referentials, referential_paths

print("chemins fictomed :", referential_paths(REFERENTIALS))
etat = []
if present(REFERENTIALS, "dossier de référentiels AP-HP complet (livraison des référentiels, mai 2026)"):
    try:
        # paresseux pour les parquets ; les CSV (chu…) sont vérifiés dès la construction
        refs = load_referentials(REFERENTIALS)
    except FileNotFoundError as exc:
        refs = {}
        print("le loader fictomed refuse le dossier (fichier manquant) :", exc,
              "\n         → obtenir le dossier de référentiels complet")
    for cle, lz in refs.items():
        try:
            apercu = lz.head(1).collect()
            etat.append({"referentiel": cle, "etat": "OK", "colonnes": ", ".join(apercu.columns)[:70]})
        except Exception as exc:  # fichier absent ou illisible : dit, pas levé
            etat.append({"referentiel": cle, "etat": f"ABSENT/ILLISIBLE ({type(exc).__name__})", "colonnes": ""})
    if etat:
        display(pl.DataFrame(etat))

if any(e["referentiel"] == "hospitals" and e["etat"] == "OK" for e in etat):
    hopitaux = refs["hospitals"].collect()
    print(f"chu : {hopitaux.height} établissements — l'hôpital qui signe le CRH est tiré ici")
    display(hopitaux.head(5))
if any(e["referentiel"] == "specialty" and e["etat"] == "OK" for e in etat):
    print("specialty : spécialité d'unité par racine de GHM et classe d'âge (détail fiche 3)")
    display(refs["specialty"].head(3).collect())

In [ ]:
# Exploration — référentiels fictomed

## 7. Le journal `usage_log.csv`

- **Rôle** : journal d'**observation** des appels Mistral — une ligne par
  scénario et par run réel, tous tests confondus (tokens, coût, transport,
  run partiel). `usage.json` de chaque test reste la source des coûts
  (`summarize_costs`) ; ce CSV sert à l'analyse transversale.
- **Producteur, évolution** : `bench.generate` l'alimente en append à
  chaque run réel (spec §7) ; jamais réécrit.
- **Emplacement** : `generation/usage_log.csv` (versionné).
- **Ce qui fait foi** : `bench/generate.py` (colonnes écrites) ;
  l'analyse détaillée est dans `generation/notebook_bilan_api.ipynb` —
  **non dupliquée ici** : existence, période couverte, nombre de runs.
- **Consommation** : `bilan()` du notebook de génération (stats rapides),
  `notebook_bilan_api.ipynb`.

In [ ]:
if present(USAGE_LOG, "créé par le premier run réel de bench.generate (aucun run réel encore sur ce clone)"):
    log = pl.read_csv(USAGE_LOG, schema_overrides={"test": pl.String, "scenario": pl.String, "batch_id": pl.String})
    print(f"{log.height} ligne(s) — colonnes : {log.columns}")
    print(f"période couverte : {log['timestamp_utc'].min()} → {log['timestamp_utc'].max()}")
    print(f"runs : {log['batch_id'].n_unique()} — tests : {sorted(log['test'].unique().to_list())} — "
          f"sorties : {sorted(log['out'].unique().to_list())} — runs partiels : {int(log['partial'].sum())} ligne(s)")
    display(log.group_by("test", "out").agg(pl.len().alias("scenarios"), pl.col("batch_id").n_unique().alias("runs")).sort(["test", "out"]))
    print("Analyse complète (coûts, projection, évolution run par run) : generation/notebook_bilan_api.ipynb")

In [ ]:
# Exploration — usage_log

## 8. Anatomie d'un dossier de run

- **Rôle** : ce que l'on trouve dans un scénario généré — le dossier
  `generation/runs/<NN>/<scénario>/` est autonome : ses prompts, son
  figement, ses sorties. La chaîne des tests est la chaîne des versions du
  jeu de templates (`system/one_gen/`).
- **Producteur, évolution** : `bench` (seeding, figement, génération,
  vérificateur) piloté par `notebook_generation_bench.ipynb` ; un test ne
  se nettoie jamais, on crée `runs/NN+1`.
- **Emplacement** : `generation/runs/` (versionné et partagé — spec §9,
  sauf `batches/`, `.fictomed/`, `.preview/` ignorés).
- **Ce qui fait foi** : `docs/spec_testrun_run_stage.md` §2 (arborescence
  d'un test, `test.json`), §3 (API), §7 (`usage.json`).
- **Consommation** : `generate` découvre les dossiers scénario
  (`scenario_dirs`) et y lit / écrit les fichiers ci-dessous.

In [ ]:
DESCRIPTIONS = {
    "template.txt": "famille clinique du scénario (stem du template fictomed) — clé du figement",
    "user_generation.txt": "user prompt fictomed : scénario (patient, séjour, codes, fiches) + bloc contexte (enrichissement)",
    "prefix.txt": "prefill assistant (prefix du jeu system/one_gen/prefix.txt, sinon celui de fictomed)",
    "prompt_system_one_gen.txt": "prompt système FIGÉ : copie de system/one_gen/<famille>.txt au figement",
    "crh_generation.txt": "sortie du run réel (JSON : CR + dictionnaire de formulations) — écrasée à chaque re-run",
    "crh_generation.md": "aperçu lisible du CRH (scripts/show_crh.py --md), versionné avec le test",
    "prompt_system_verif.txt": "prompt système du vérificateur (write_prompts, optionnel)",
    "user_verification.txt": "user prompt du vérificateur",
    "verdict.txt": "sortie du vérificateur (run réel optionnel)",
    "user_regeneration.txt": "user prompt de régénération (scripts/prepare_regeneration.py) — dossiers rejetés seulement",
}
RACINE_TEST = {
    "test.json": "provenance de la graine (generation_ids, templates, fictomed_version/commit, notes) — spec §2.2",
    "usage.json": "journal append-only des runs réels (tokens, coût, transport) — spec §7",
    "system/one_gen/": "LE JEU du test : un .txt par famille + prefix.txt + regles_atih.yml — l'objet versionné",
    "batches/": "JSONL techniques des appels (ignoré par git)",
    ".fictomed/": "fichiers de travail fictomed : servers.yaml, tirage archivé, sorties (ignoré par git)",
}

if present(TESTS_DIR, "créé par le premier test du notebook de génération (montage + seeding)"):
    tests = sorted(d.name for d in TESTS_DIR.iterdir() if d.is_dir() and not d.name.startswith("."))
    td = TESTS_DIR / tests[-1]
    noms = scenario_dirs(td)
    print(f"tests : {tests} — le plus récent : {td.name} ({len(noms)} dossiers scénario)\n")
    print("--- racine du test ---")
    for nom, role in RACINE_TEST.items():
        p = td / nom
        print(f"  {'OK ' if p.exists() else '-- '} {nom:28s} {role}")
    if noms:
        sd = td / noms[0]
        print(f"\n--- dossier scénario {td.name}/{noms[0]} ---")
        for f in sorted(sd.iterdir()):
            role = DESCRIPTIONS.get(f.name, "(fichier libre : itération, copie, variante)")
            print(f"  {f.name:28s} {f.stat().st_size:7d} o   {role}")
        manquants = [n for n in DESCRIPTIONS if not (sd / n).exists()]
        print("  non présents ici :", manquants)
        print(f"\n  template.txt : {(sd / 'template.txt').read_text(encoding='utf-8').strip()!r}")
        if (sd / "prefix.txt").is_file():
            print(f"  prefix.txt   : {(sd / 'prefix.txt').read_text(encoding='utf-8')!r}")
        print("  user_generation.txt (début) :")
        print("    " + "\n    ".join((sd / "user_generation.txt").read_text(encoding="utf-8").splitlines()[:12]))
    else:
        print("(pas encore de dossier scénario dans ce test)")

In [ ]:
# Exploration — dossier de run